# 01 — Data Generation

## Phase 1: Simulating the Stochastic Heat Equation

This notebook generates synthetic volatility surface data by simulating the
stochastic PDE from **Slide 9** of Ioselevich's presentation:

$$\partial_t p(z,t) = D \, \partial_z^2 \, p(z,t) + \xi(z,t)$$

where:
- $p(z,t)$ is the perturbation of the vol surface at moneyness $z$, time $t$
- $D$ is the diffusion coefficient ("stiffness")
- $\xi(z,t)$ is spatiotemporal white noise

In Fourier space, each mode evolves independently as an **Ornstein-Uhlenbeck process**:

$$\partial_t f_k = -D k^2 f_k + \xi_k(t)$$

with stationary variance $\langle f_k^2 \rangle \sim 1/k^2$.

### Why synthetic data?

Real options data is expensive. By generating from the known model, we can:
1. Verify every step of our pipeline against ground truth
2. Understand what the PCA *should* find before looking at real data
3. Tune parameters and build intuition

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')

from src.simulate import StochasticHeatEquation, SPDEConfig, generate_synthetic_dataset
from src.plotting import apply_style, COLORS

apply_style()
print('Setup complete.')

## 1.1 — Configure and run the simulation

We simulate ~8 years of daily data (2000 trading days) on a moneyness grid
$z \in [-3, 3]$ with 50 grid points. The diffusion coefficient $D = 0.05$
is chosen to give realistic eigenvalue decay.

In [ ]:
# Configure the simulation
config = SPDEConfig(
    D=0.05,              # diffusion coefficient (stiffness)
    n_z=50,              # spatial grid points
    z_min=-3.0,          # deep OTM puts
    z_max=3.0,           # deep OTM calls
    n_modes=25,          # Fourier modes
    noise_amplitude=1.0, # noise strength
    seed=42,             # reproducibility
)

spde = StochasticHeatEquation(config)
print(f'Domain: z ∈ [{config.z_min}, {config.z_max}], L = {spde.L}')
print(f'Grid points: {config.n_z}')
print(f'Fourier modes: {config.n_modes}')
print(f'Diffusion D = {config.D}')
print(f'\nDecay rates (first 5 modes): {spde.decay_rates[:5].round(4)}')
print(f'Stationary variance (first 5 modes): {spde.stationary_variance[:5].round(4)}')

In [ ]:
# Run the simulation
n_days = 2000
surface = spde.simulate(n_days)
returns = spde.compute_returns(surface)

print(f'Surface shape: {surface.shape}  (days × grid points)')
print(f'Returns shape: {returns.shape}')
print(f'Surface std (mean across grid): {surface.std(axis=0).mean():.4f}')
print(f'Returns std (mean across grid): {returns.std(axis=0).mean():.4f}')

## 1.2 — Visualise the simulated surface

Think of this as watching a vibrating string over time. Each row is one
day's snapshot of the surface perturbation.

In [ ]:
# Heatmap of the surface over time
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(
    surface[:500].T,
    aspect='auto',
    cmap='RdBu_r',
    extent=[0, 500, config.z_min, config.z_max],
    origin='lower',
)
plt.colorbar(im, ax=ax, label='perturbation p(z,t)')
ax.set_xlabel('trading day')
ax.set_ylabel('moneyness z')
ax.set_title('Simulated Volatility Surface Perturbation (first 500 days)')
plt.tight_layout()
plt.show()

In [ ]:
# A few daily snapshots — the "vibrating string" picture (Slide 9)
fig, ax = plt.subplots(figsize=(10, 4))
for day in [0, 50, 100, 200, 500]:
    ax.plot(spde.z_grid, surface[day], label=f'day {day}', alpha=0.8)

ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
ax.set_xlabel('moneyness z')
ax.set_ylabel('displacement from equilibrium')
ax.set_title('Surface Snapshots — Vibrating String Analogy')
ax.legend()
plt.tight_layout()
plt.show()

## 1.3 — Check theoretical properties

Before running PCA, let's verify the simulation produces the expected
statistical properties:

1. The variance of returns at each grid point
2. The correlation between neighbouring points (should decay with distance)
3. The autocorrelation of returns at a single point

In [ ]:
# Variance profile across the grid
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Variance of surface levels
axes[0].plot(spde.z_grid, surface.var(axis=0), 'o-', markersize=3)
axes[0].set_xlabel('moneyness z')
axes[0].set_ylabel('variance')
axes[0].set_title('Variance of Surface Level')

# Variance of returns
axes[1].plot(spde.z_grid, returns.var(axis=0), 'o-', markersize=3, color=COLORS['mode1'])
axes[1].set_xlabel('moneyness z')
axes[1].set_ylabel('variance')
axes[1].set_title('Variance of Daily Returns')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation vs distance
corr_matrix = np.corrcoef(returns.T)
n_z = len(spde.z_grid)
mid = n_z // 2  # reference point at z ≈ 0

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Full correlation matrix
im = axes[0].imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=axes[0])
axes[0].set_title('Return Correlation Matrix')
axes[0].set_xlabel('grid point')
axes[0].set_ylabel('grid point')

# Correlation with midpoint as function of distance
distances = np.abs(spde.z_grid - spde.z_grid[mid])
axes[1].plot(distances, corr_matrix[mid, :], 'o-', markersize=3)
axes[1].set_xlabel('|z - z_mid|')
axes[1].set_ylabel('correlation')
axes[1].set_title('Correlation vs Distance from Midpoint')

plt.tight_layout()
plt.show()

## 1.4 — Save the dataset

Save the synthetic data for use in subsequent notebooks.

In [ ]:
import os

output_dir = '../data/synthetic'
os.makedirs(output_dir, exist_ok=True)

np.savez(
    os.path.join(output_dir, 'spde_dataset.npz'),
    z_grid=spde.z_grid,
    surface=surface,
    returns=returns,
    D=config.D,
    n_modes=config.n_modes,
    noise_amplitude=config.noise_amplitude,
)

print(f'Dataset saved to {output_dir}/spde_dataset.npz')
print(f'File size: {os.path.getsize(os.path.join(output_dir, "spde_dataset.npz")) / 1e6:.1f} MB')

## 1.5 — Next steps

With our synthetic data generated, we're ready to:

1. **Notebook 02**: Apply the (z, τ) reparameterization (already in z-space,
   but we'll implement the τ transform for the maturity dimension)
2. **Notebook 03**: Run PCA and discover the eigenmode structure
3. **Notebook 04**: Compare PCA results to theoretical Fourier modes

The key question for Notebook 03: does PCA on our simulated returns
recover sinusoidal eigenmodes with $\sim k^{-2}$ eigenvalue decay?